# Read Bronze data - Create data frame

In [2]:
# Load messy raw data into DataFrames from CSVs
df_orders_raw = spark.read.parquet("abfss://Reatil_data_Project@onelake.dfs.fabric.microsoft.com/medallion.Lakehouse/Files/Bronze/orders_data.parquet")
df_returns_raw = spark.read.parquet("abfss://Reatil_data_Project@onelake.dfs.fabric.microsoft.com/medallion.Lakehouse/Files/Bronze/returns_data.xlsx.parquet")
df_inventory_raw = spark.read.parquet("abfss://Reatil_data_Project@onelake.dfs.fabric.microsoft.com/medallion.Lakehouse/Files/Bronze/inventory_data.parquet")


StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 4, Finished, Available, Finished, False)

In [4]:
display(df_orders_raw)

StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6cb29ccc-1488-40c2-9a6e-f56593f4f7c6)

# Handle First row for inventory data frame files

In [6]:
# Extract first row as header
first_row=df_returns_raw.first()
columns=[str(item).strip() for item in first_row]

#remove the first row (header row row now part of data)
df_returns_raw=df_returns_raw.rdd.zipWithIndex().filter(lambda x:x[1]>0).map(lambda x:x[0]).toDF(columns)

#show cleaned columns
display(df_returns_raw)

StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 594d1cbe-6d9f-4dec-b1e6-03940a8615e0)

In [7]:
display(df_returns_raw)

StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 71268402-8781-429b-8ff3-a6b7c4a45e54)

In [3]:
display(df_inventory_raw)

StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb1b21a5-a35c-4efa-b38e-d43ddbcba54f)

# Create Bronze delta tables

In [8]:
# Save raw data to Bronze tables without any transformations
df_orders_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_orders")
df_returns_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_returns")
df_inventory_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_inventory")


StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 10, Finished, Available, Finished, False)

In [9]:
%%sql
select * from bronze_inventory

StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 11, Finished, Available, Finished, False)

<Spark SQL result set with 15 rows and 7 fields>

# Clean the data - Silver Layer Preparation

In [10]:
display(df_orders_raw.limit(5))

StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72448323-46e3-42a1-8798-f54751624a26)

In [25]:

# ============================================================
# SILVER LAYER: CLEAN ORDERS, RETURNS, AND INVENTORY
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# ------------------------------------------------------------
# A. ORDERS
# ------------------------------------------------------------

df_orders_raw = spark.read.table("bronze_orders")

df_orders_silver = (
    df_orders_raw

    # Standardize column names
    .withColumnRenamed("Order_ID", "order_id")
    .withColumnRenamed("cust_id", "customer_id")
    .withColumnRenamed("Product_Name", "product_name")
    .withColumnRenamed("Qty", "quantity")
    .withColumnRenamed("Order_Date", "order_date")
    .withColumnRenamed("Order_Amount$", "order_amount")
    .withColumnRenamed("Delivery_Status", "delivery_status")
    .withColumnRenamed("Payment_Mode", "payment_mode")
    .withColumnRenamed("Ship_Address", "ship_address")
    .withColumnRenamed("Email", "email")
    .withColumnRenamed("Promo_Code", "promo_code")
    .withColumnRenamed("Feedback_Score", "feedback_score")

    # Normalize identifiers and product names
    .withColumn("order_id", F.trim(F.col("order_id").cast("string")))
    .withColumn("customer_id", F.upper(F.trim(F.col("customer_id").cast("string"))))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))

    # Convert quantity words and numeric strings
    .withColumn(
        "quantity",
        F.when(
            F.lower(F.trim(F.col("quantity").cast("string"))) == "one", 1
        )
        .when(
            F.lower(F.trim(F.col("quantity").cast("string"))) == "two", 2
        )
        .when(
            F.lower(F.trim(F.col("quantity").cast("string"))) == "three", 3
        )
        .otherwise(
            F.regexp_extract(
                F.col("quantity").cast("string"), r"^\s*(\d+)\s*$", 1
            ).cast(IntegerType())
        )
    )

    # Parse multiple date formats
    .withColumn(
        "order_date",
        F.coalesce(
            F.to_date(F.col("order_date"), "yyyy-MM-dd"),
            F.to_date(F.col("order_date"), "yyyy/MM/dd"),
            F.to_date(F.col("order_date"), "yyyy.MM.dd"),
            F.to_date(F.col("order_date"), "dd-MM-yyyy"),
            F.to_date(F.col("order_date"), "dd/MM/yyyy"),
            F.to_date(F.col("order_date"), "MM/dd/yyyy")
        )
    )

    # Clean order amounts; source values are expected to be USD-like
    .withColumn(
        "order_amount",
        F.regexp_replace(
            F.col("order_amount").cast("string"),
            r"[$,\s]",
            ""
        ).cast(DoubleType())
    )

    # Validate email
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn(
        "email",
        F.when(
            F.col("email").rlike(
                r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
            ),
            F.col("email")
        )
    )

    # Clean other fields
    .withColumn(
        "delivery_status",
        F.when(
            F.col("delivery_status").isNull() |
            (F.trim(F.col("delivery_status")) == ""),
            "unknown"
        ).otherwise(F.lower(F.trim(F.col("delivery_status"))))
    )
    .withColumn(
        "payment_mode",
        F.when(
            F.col("payment_mode").isNull() |
            (F.trim(F.col("payment_mode")) == ""),
            "unknown"
        ).otherwise(F.lower(F.trim(F.col("payment_mode"))))
    )
    .withColumn(
        "ship_address",
        F.initcap(F.trim(
            F.regexp_replace(F.col("ship_address"), r"[^a-zA-Z0-9\s]", " ")
        ))
    )
    .withColumn("promo_code", F.upper(F.trim(F.col("promo_code"))))
    .withColumn("feedback_score", F.col("feedback_score").cast(DoubleType()))

    # Remove invalid keys and missing product names
    .filter(
        F.col("order_id").isNotNull() &
        (F.col("order_id") != "") &
        F.col("customer_id").isNotNull() &
        F.col("product_name").isNotNull() &
        (F.trim(F.col("product_name")) != "")
    )

    # Retain one row per order ID
    .dropDuplicates(["order_id"])
)

# ------------------------------------------------------------
# B. RETURNS
# ------------------------------------------------------------

df_returns_raw = spark.read.table("bronze_returns")

df_returns_silver = (
    df_returns_raw

    .withColumnRenamed("Return_ID", "return_id")
    .withColumnRenamed("Order_ID", "order_id")
    .withColumnRenamed("Customer_ID", "customer_id")
    .withColumnRenamed("Product", "product_name")
    .withColumnRenamed("Return_Reason", "return_reason")
    .withColumnRenamed("Return_Date", "return_date")
    .withColumnRenamed("Refund_Status", "refund_status")
    .withColumnRenamed("Pickup_Address", "pickup_address")
    .withColumnRenamed("Return_Amount", "return_amount")

    # Excel can represent order IDs as numeric values such as 1001.0
    .withColumn(
        "order_id",
        F.regexp_extract(
            F.col("order_id").cast("string"), r"(\d+)", 1
        )
    )
    .withColumn("return_id", F.trim(F.col("return_id").cast("string")))
    .withColumn("customer_id", F.upper(F.trim(F.col("customer_id"))))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))

    # Parse return dates
    .withColumn(
        "return_date",
        F.coalesce(
            F.to_date(F.col("return_date"), "yyyy-MM-dd"),
            F.to_date(F.col("return_date"), "yyyy/MM/dd"),
            F.to_date(F.col("return_date"), "yyyy.MM.dd"),
            F.to_date(F.col("return_date"), "dd-MM-yyyy"),
            F.to_date(F.col("return_date"), "dd/MM/yyyy"),
            F.to_date(F.col("return_date"), "MM/dd/yyyy")
        )
    )

    .withColumn(
        "refund_status",
        F.lower(F.trim(
            F.regexp_replace(F.col("refund_status"), r"[^a-zA-Z0-9\s]", "")
        ))
    )

    # Keep the numeric amount but do not assume all currencies are equivalent
    .withColumn(
        "return_amount",
        F.regexp_replace(
            F.regexp_replace(
                F.col("return_amount").cast("string"),
                r"(?i)INR|USD|Rs\.?",
                ""
            ),
            r"[₹$,\s]",
            ""
        ).cast(DoubleType())
    )

    .withColumn(
        "pickup_address",
        F.initcap(F.trim(
            F.regexp_replace(F.col("pickup_address"), r"[^a-zA-Z0-9\s]", " ")
        ))
    )

    .filter(
        F.col("return_id").isNotNull() &
        (F.col("return_id") != "") &
        F.col("order_id").isNotNull() &
        (F.col("order_id") != "")
    )
    .dropDuplicates(["return_id"])
)

# ------------------------------------------------------------
# C. INVENTORY
# ------------------------------------------------------------

df_inventory_raw = spark.read.table("bronze_inventory")

df_inventory_silver = (
    df_inventory_raw

    .withColumnRenamed("product_id", "product_id")
    .withColumnRenamed("productName", "product_name")
    .withColumnRenamed("stock", "stock")
    .withColumnRenamed("last_stocked", "last_stocked")
    .withColumnRenamed("warehouse", "warehouse")
    .withColumnRenamed("cost_price", "cost_price")
    .withColumnRenamed("available", "available")

    .withColumn("product_id", F.trim(F.col("product_id").cast("string")))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))

    # Convert numeric stock and supported number words
    .withColumn(
        "stock",
        F.when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "one", 1
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "two", 2
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "three", 3
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "twelve", 12
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "fifteen", 15
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "eighteen", 18
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "twenty", 20
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))) == "twenty five", 25
        )
        .when(
            F.lower(F.trim(F.col("stock").cast("string"))).isin("", "null"),
            F.lit(None).cast(IntegerType())
        )
        .otherwise(
            F.regexp_extract(
                F.col("stock").cast("string"), r"(\d+)", 1
            ).cast(IntegerType())
        )
    )

    # Parse last-stocked dates
    .withColumn(
        "last_stocked",
        F.coalesce(
            F.to_date(F.col("last_stocked"), "yyyy-MM-dd"),
            F.to_date(F.col("last_stocked"), "yyyy/MM/dd"),
            F.to_date(F.col("last_stocked"), "yyyy.MM.dd"),
            F.to_date(F.col("last_stocked"), "dd-MM-yyyy"),
            F.to_date(F.col("last_stocked"), "dd/MM/yyyy"),
            F.to_date(F.col("last_stocked"), "dd.MM.yyyy"),
            F.to_date(F.col("last_stocked"), "MM-dd-yyyy"),
            F.to_date(F.col("last_stocked"), "MM/dd/yyyy")
        )
    )

    # Remove currency labels and separators from cost
    .withColumn(
        "cost_price",
        F.regexp_replace(
            F.regexp_replace(
                F.col("cost_price").cast("string"),
                r"(?i)INR|USD|Rs\.?",
                ""
            ),
            r"[₹$,\s]",
            ""
        ).cast(DoubleType())
    )

    .withColumn(
        "warehouse",
        F.initcap(F.trim(
            F.regexp_replace(F.col("warehouse"), r"[^a-zA-Z0-9\s]", " ")
        ))
    )

    .withColumn(
        "available",
        F.when(
            F.lower(F.trim(F.col("available"))).isin("yes", "y", "true"),
            F.lit(True)
        )
        .when(
            F.lower(F.trim(F.col("available"))).isin("no", "n", "false"),
            F.lit(False)
        )
        .otherwise(F.lit(None).cast("boolean"))
    )

    .filter(
        F.col("product_id").isNotNull() &
        F.col("product_name").isNotNull()
    )
    .dropDuplicates(["product_id"])
)

# ------------------------------------------------------------
# SAVE SILVER DELTA TABLES
# ------------------------------------------------------------

df_orders_silver.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("silver_orders")

df_returns_silver.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("silver_returns")

df_inventory_silver.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("silver_inventory")

print("Silver layer completed successfully.")

display(df_orders_silver)
display(df_returns_silver)
display(df_inventory_silver)


StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 27, Finished, Available, Finished, False)

Silver layer completed successfully.


SynapseWidget(Synapse.DataFrame, 47c8c696-29d0-44cc-aec0-9305d5cffc63)

SynapseWidget(Synapse.DataFrame, df05f497-81fb-4877-b1e9-d3fd51d99717)

SynapseWidget(Synapse.DataFrame, 8d5de5dd-ba24-4197-becb-ae66143339e1)

# GOLD LAYER – Aggregation & KPIs


In [26]:

# ============================================================
# GOLD LAYER: PRODUCT-LEVEL BUSINESS KPIs
# ============================================================

from pyspark.sql import functions as F

# Load Silver Delta tables
df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")

# ------------------------------------------------------------
# STEP 1: Create a consistent product key
# ------------------------------------------------------------

df_orders = df_orders.withColumn(
    "product_key", F.lower(F.trim(F.col("product_name")))
)

df_returns = df_returns.withColumn(
    "product_key", F.lower(F.trim(F.col("product_name")))
)

df_inventory = df_inventory.withColumn(
    "product_key", F.lower(F.trim(F.col("product_name")))
)

# ------------------------------------------------------------
# STEP 2: Aggregate order KPIs by product
# ------------------------------------------------------------

df_order_kpi = (
    df_orders
    .groupBy("product_key")
    .agg(
        F.first("product_name", ignorenulls=True).alias("Product_Name"),
        F.countDistinct("order_id").alias("Total_Orders"),
        F.countDistinct("customer_id").alias("Unique_Customers"),
        F.sum("quantity").alias("Total_Quantity_Sold"),
        F.round(F.sum("order_amount"), 2).alias("Total_Revenue"),
        F.round(F.avg("order_amount"), 2).alias("Avg_Order_Value")
    )
)

# ------------------------------------------------------------
# STEP 3: Link returns to the products in Orders
# Use the order ID as the primary link
# ------------------------------------------------------------

df_return_links = (
    df_returns.alias("r")
    .join(
        df_orders.select("order_id", "product_key").distinct().alias("o"),
        on="order_id",
        how="left"
    )
    .select(
        "order_id",
        F.coalesce(
            F.col("o.product_key"),
            F.col("r.product_key")
        ).alias("product_key"),
        F.col("r.return_id")
    )
)

# Count return records and returned orders separately
df_return_kpi = (
    df_return_links
    .filter(F.col("return_id").isNotNull())
    .groupBy("product_key")
    .agg(
        F.countDistinct("return_id").alias("Total_Returns"),
        F.countDistinct("order_id").alias("Returned_Orders")
    )
)

# ------------------------------------------------------------
# STEP 4: Aggregate inventory by product
# ------------------------------------------------------------

df_inventory_kpi = (
    df_inventory
    .groupBy("product_key")
    .agg(
        F.sum("stock").alias("Total_Stock"),
        F.countDistinct("product_id").alias("Inventory_Records"),
        F.sum(
            F.when(F.col("available") == True, 1).otherwise(0)
        ).alias("Available_Inventory_Records")
    )
)

# ------------------------------------------------------------
# STEP 5: Join the product-level aggregates
# ------------------------------------------------------------

df_gold = (
    df_order_kpi
    .join(df_return_kpi, on="product_key", how="left")
    .join(df_inventory_kpi, on="product_key", how="full")
)

# Fill missing counts with zero
for c in [
    "Total_Orders",
    "Unique_Customers",
    "Total_Quantity_Sold",
    "Total_Returns",
    "Returned_Orders",
    "Total_Stock",
    "Inventory_Records",
    "Available_Inventory_Records"
]:
    df_gold = df_gold.withColumn(
        c, F.coalesce(F.col(c), F.lit(0))
    )

# ------------------------------------------------------------
# STEP 6: Calculate return rate
# ------------------------------------------------------------

df_gold = df_gold.withColumn(
    "Return_Rate_Pct",
    F.when(
        F.col("Total_Orders") > 0,
        F.round(
            F.col("Returned_Orders") /
            F.col("Total_Orders") * 100,
            2
        )
    ).otherwise(F.lit(0.0))
)

# ------------------------------------------------------------
# STEP 7: Select final Gold columns
# ------------------------------------------------------------

df_gold = df_gold.select(
    "product_key",
    "Product_Name",
    "Total_Orders",
    "Unique_Customers",
    "Total_Quantity_Sold",
    "Total_Revenue",
    "Avg_Order_Value",
    "Total_Returns",
    "Returned_Orders",
    "Return_Rate_Pct",
    "Total_Stock",
    "Inventory_Records",
    "Available_Inventory_Records"
)

# ------------------------------------------------------------
# STEP 8: Validate and save Gold Delta table
# ------------------------------------------------------------

print("Gold product KPI table:")
df_gold.printSchema()
display(df_gold)

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_product_kpis")

print("Gold layer completed successfully.")


StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 28, Finished, Available, Finished, False)

Gold product KPI table:
root
 |-- product_key: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Total_Orders: long (nullable = false)
 |-- Unique_Customers: long (nullable = false)
 |-- Total_Quantity_Sold: long (nullable = false)
 |-- Total_Revenue: double (nullable = true)
 |-- Avg_Order_Value: double (nullable = true)
 |-- Total_Returns: long (nullable = false)
 |-- Returned_Orders: long (nullable = false)
 |-- Return_Rate_Pct: double (nullable = true)
 |-- Total_Stock: long (nullable = false)
 |-- Inventory_Records: long (nullable = false)
 |-- Available_Inventory_Records: long (nullable = false)



SynapseWidget(Synapse.DataFrame, a6efb440-693f-48b8-b7f9-0fdb5d3e88a5)

Gold layer completed successfully.


In [27]:

# Check tables and row counts

tables_to_check = [
    "bronze_orders",
    "bronze_returns",
    "bronze_inventory",
    "silver_orders",
    "silver_returns",
    "silver_inventory",
    "gold_product_kpis"
]

for table_name in tables_to_check:
    df = spark.read.table(table_name)
    print(f"{table_name}: {df.count()} rows")


StatementMeta(, f7de87b5-1756-439e-9cce-5c94398e9b01, 29, Finished, Available, Finished, False)

bronze_orders: 15 rows
bronze_returns: 15 rows
bronze_inventory: 15 rows
silver_orders: 12 rows
silver_returns: 14 rows
silver_inventory: 15 rows
gold_product_kpis: 17 rows
